# Dense Retrieval — Bi-Encoder (Google Colab)

Pipeline:
1. Upload & giải nén dataset
2. Encode documents & queries bằng pretrained Sentence-Transformer
3. Tìm kiếm bằng FAISS (cosine similarity)
4. Evaluate F1-score

**Model**: Weights đóng băng (Freeze) — chỉ inference, không train.

## 1. Cài đặt thư viện

In [1]:
!pip install sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.2 MB/s eta 0:00:00


## 2. Upload & Giải nén dataset

Upload file `dataset_nlp.zip` (chứa folder `data/` với Cranfield corpus + queries + answers).

In [2]:
import zipfile
import os

# Upload file từ máy tính
from google.colab import files
uploaded = files.upload()  # Chọn file dataset_nlp.zip

# Giải nén
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('.')
    print(f"Extracted {len(z.namelist())} files from {zip_name}")

# Tự động tìm thư mục data
# Refined logic to find the correct data directory.
# We are looking for a directory that contains both the CSV files and the 'Cranfield' subdirectory.
DATA_DIR = None
for root, dirs, files in os.walk('.'):
    if 'public_test_queries.csv' in files and 'public_test_answers.csv' in files and 'Cranfield' in dirs:
        DATA_DIR = root
        break

# Fallback if DATA_DIR is still not found (e.g., different directory structure)
if DATA_DIR is None:
    # Try to infer from zip_name, removing the '.zip' extension and potential '(1)' suffix
    base_extracted_name = os.path.splitext(zip_name)[0]
    if base_extracted_name.endswith(' (1)'):
        base_extracted_name = base_extracted_name[:-4] # Remove ' (1)'

    # Check for the base folder name directly
    if os.path.isdir(base_extracted_name):
        DATA_DIR = base_extracted_name
    elif os.path.isdir(f'./{base_extracted_name}'): # Sometimes it's `./folder_name`
        DATA_DIR = f'./{base_extracted_name}'
    else:
        print("Warning: Could not automatically determine DATA_DIR. Defaulting to './dataset_nlp' as a common name.")
        DATA_DIR = './dataset_nlp' # Last resort, assuming common folder name


CORPUS_DIR = os.path.join(DATA_DIR, 'Cranfield')
QUERY_CSV = os.path.join(DATA_DIR, 'public_test_queries.csv')
ANSWER_CSV = os.path.join(DATA_DIR, 'public_test_answers.csv')

print(f"\nData directory: {DATA_DIR}")
print(f"Corpus dir   : {CORPUS_DIR} (exists={os.path.exists(CORPUS_DIR)})")
print(f"Query CSV    : {QUERY_CSV} (exists={os.path.exists(QUERY_CSV)})")
print(f"Answer CSV   : {ANSWER_CSV} (exists={os.path.exists(ANSWER_CSV)})")

Saving dataset_nlp.zip to dataset_nlp.zip
Extracted 2808 files from dataset_nlp.zip

Data directory: ./dataset_nlp
Corpus dir   : ./dataset_nlp/Cranfield (exists=True)
Query CSV    : ./dataset_nlp/public_test_queries.csv (exists=True)
Answer CSV   : ./dataset_nlp/public_test_answers.csv (exists=True)


## 3. Utility Functions (inline)

Các hàm load data và evaluate — inline để chạy trên Colab không cần folder `utils/`.

In [3]:
import csv
from typing import Dict, List, Tuple

# ===================== DATA LOADER =====================

def load_corpus(corpus_dir: str) -> Dict[int, str]:
    """Load all {docID}.txt files from corpus_dir."""
    corpus = {}
    for fname in os.listdir(corpus_dir):
        if fname.endswith('.txt'):
            try:
                doc_id = int(fname.replace('.txt', ''))
                with open(os.path.join(corpus_dir, fname), 'r',
                          encoding='utf-8', errors='ignore') as f:
                    corpus[doc_id] = f.read()
            except ValueError:
                pass
    return corpus

def load_queries(query_csv: str) -> Dict[int, str]:
    """Load public_test_queries.csv."""
    queries = {}
    with open(query_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            queries[int(row['query_id'])] = row['query']
    return queries

def load_answers(answer_csv: str) -> Dict[int, List[int]]:
    """Load public_test_answers.csv."""
    answers = {}
    with open(answer_csv, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            qid = int(row['query_id'])
            doc_ids = [int(x) for x in row['relevant_docIDs'].split() if x.strip()]
            answers[qid] = doc_ids
    return answers

def save_submission(results: Dict[int, List[int]], output_path: str):
    """Save results as nlp_submission.csv."""
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['query_id', 'relevant_docIDs'])
        for qid in sorted(results.keys()):
            writer.writerow([qid, ' '.join(map(str, results[qid]))])
    print(f'Saved submission → {output_path}')

# ===================== EVALUATE =====================

def precision_recall_f1(predicted: List[int], relevant: List[int]) -> Tuple[float, float, float]:
    pred_set = set(predicted)
    rel_set = set(relevant)
    tp = len(pred_set & rel_set)
    p = tp / len(pred_set) if pred_set else 0.0
    r = tp / len(rel_set) if rel_set else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1

def evaluate(results, answers, verbose=True):
    precisions, recalls, f1s = [], [], []
    for qid, relevant in answers.items():
        predicted = results.get(qid, [])
        p, r, f1 = precision_recall_f1(predicted, relevant)
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)
    macro_p = sum(precisions) / len(precisions)
    macro_r = sum(recalls) / len(recalls)
    macro_f1 = sum(f1s) / len(f1s)
    metrics = {'precision': macro_p, 'recall': macro_r, 'f1': macro_f1}
    if verbose:
        print(f'  Precision : {macro_p:.4f}')
        print(f'  Recall    : {macro_r:.4f}')
        print(f'  F1        : {macro_f1:.4f}')
    return metrics

print('Utility functions loaded ✓')

Utility functions loaded ✓


## 4. Load Data

In [4]:
corpus  = load_corpus(CORPUS_DIR)
queries = load_queries(QUERY_CSV)
answers = load_answers(ANSWER_CSV)

# Sắp xếp corpus theo doc_id
doc_ids = sorted(corpus.keys())
doc_texts = [corpus[did] for did in doc_ids]

print(f'Corpus : {len(corpus)} documents')
print(f'Queries: {len(queries)} queries')
print(f'Answers: {len(answers)} queries with ground truth')
print()
for qid, qtext in queries.items():
    print(f'  Query {qid}: {qtext[:100]}...')
    print(f'    Relevant docs: {answers[qid]}')

Corpus : 1400 documents
Queries: 3 queries
Answers: 3 queries with ground truth

  Query 62: how far around a cylinder and under what conditions of flow,  if any, is the velocity just outside o...
    Relevant docs: [56, 567, 1084]
  Query 160: compressive circumferential stresses in a torispherical shell reveal the possibility of buckling und...
    Relevant docs: [1134, 1137, 1138]
  Query 215: is it possible to predict the shape of a shroud which will allow simulation of the nose region flow ...
    Relevant docs: [37, 35]


## 5. Load Pretrained Model

| Model | Kích thước | Ghi chú |
|-------|-----------|--------|
| `all-mpnet-base-v2` | 420MB | General purpose |
| `BAAI/bge-base-en-v1.5` | 440MB | Mạnh cho retrieval |
| `BAAI/bge-large-en-v1.5` | 1.3GB | Mạnh nhất |
| `intfloat/e5-base-v2` | 440MB | Cần prefix "query:" / "passage:" |
| `facebook/contriever` | 440MB | Meta's retrieval model |

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

# ============================================================
# CHỌN MODEL TẠI ĐÂY
# ============================================================
MODEL_NAME = 'all-mpnet-base-v2'
# MODEL_NAME = 'BAAI/bge-base-en-v1.5'
# MODEL_NAME = 'intfloat/e5-base-v2'

# E5 model cần prefix, các model khác không cần
USE_PREFIX = 'e5' in MODEL_NAME
DOC_PREFIX = 'passage: ' if USE_PREFIX else ''
QUERY_PREFIX = 'query: ' if USE_PREFIX else ''

print(f'Loading model: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME)
print(f'Model loaded! Embedding dim: {model.get_sentence_embedding_dimension()}')
if USE_PREFIX:
    print(f'Using prefix: doc="{DOC_PREFIX}" query="{QUERY_PREFIX}"')

Loading model: all-mpnet-base-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded! Embedding dim: 768


/tmp/ipykernel_4230/1224468208.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f'Model loaded! Embedding dim: {model.get_sentence_embedding_dimension()}')


## 6. Encode Corpus & Build FAISS Index

In [6]:
import faiss

# Encode documents
print(f'Encoding {len(doc_texts)} documents...')
t0 = time.time()
doc_embeddings = model.encode(
    [DOC_PREFIX + text for text in doc_texts],
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True,
)
print(f'Done! {time.time()-t0:.1f}s | Shape: {doc_embeddings.shape}')

# Build FAISS index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings.astype(np.float32))
print(f'FAISS index: {index.ntotal} vectors, {dimension}d')

Encoding 1400 documents...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Done! 18.2s | Shape: (1400, 768)
FAISS index: 1400 vectors, 768d


## 7. Search & Evaluate

In [7]:
TOP_K = 20

# Encode queries
query_ids = sorted(queries.keys())
query_embeddings = model.encode(
    [QUERY_PREFIX + queries[qid] for qid in query_ids],
    normalize_embeddings=True,
)

# Search
scores, indices = index.search(query_embeddings.astype(np.float32), TOP_K)

# Build results
dense_results = {}
for i, qid in enumerate(query_ids):
    dense_results[qid] = [doc_ids[idx] for idx in indices[i]]

# Show top results per query
for i, qid in enumerate(query_ids):
    print(f'\nQuery {qid}: {queries[qid][:80]}...')
    print(f'  Relevant: {answers.get(qid, [])}')
    print(f'  Top-5:')
    for j in range(min(20, TOP_K)):
        did = dense_results[qid][j]
        sc = scores[i][j]
        tag = ' ✅' if did in answers.get(qid, []) else ''
        print(f'    {j+1}. Doc {did} (score={sc:.4f}){tag}')

# Evaluate
print(f'\n{"="*50}')
print(f'Dense Retrieval ({MODEL_NAME}) | top_k={TOP_K}')
print(f'{"="*50}')
dense_metrics = evaluate(dense_results, answers)

for qid in sorted(answers):
    predicted = dense_results.get(qid, [])
    relevant = answers[qid]
    p, r, f1 = precision_recall_f1(predicted, relevant)
    found = [d for d in relevant if d in predicted]
    missed = [d for d in relevant if d not in predicted]
    print(f'  Query {qid}: P={p:.3f} R={r:.3f} F1={f1:.3f} | found={found} missed={missed}')


Query 62: how far around a cylinder and under what conditions of flow,  if any, is the vel...
  Relevant: [56, 567, 1084]
  Top-5:
    1. Doc 785 (score=0.7078)
    2. Doc 382 (score=0.7029)
    3. Doc 1038 (score=0.6840)
    4. Doc 105 (score=0.6583)
    5. Doc 261 (score=0.6264)
    6. Doc 784 (score=0.6254)
    7. Doc 1078 (score=0.6204)
    8. Doc 1084 (score=0.6168) ✅
    9. Doc 1382 (score=0.6092)
    10. Doc 787 (score=0.6081)
    11. Doc 375 (score=0.6065)
    12. Doc 659 (score=0.6059)
    13. Doc 754 (score=0.6035)
    14. Doc 788 (score=0.6024)
    15. Doc 23 (score=0.5939)
    16. Doc 977 (score=0.5919)
    17. Doc 117 (score=0.5909)
    18. Doc 1081 (score=0.5905)
    19. Doc 1182 (score=0.5835)
    20. Doc 1385 (score=0.5814)

Query 160: compressive circumferential stresses in a torispherical shell reveal the possibi...
  Relevant: [1134, 1137, 1138]
  Top-5:
    1. Doc 1071 (score=0.8410)
    2. Doc 1052 (score=0.7490)
    3. Doc 1134 (score=0.7323) ✅
    4. Doc 1174 (s

## 8. Lưu kết quả & Download

In [8]:
# Lưu submission
save_submission(dense_results, 'nlp_submission.csv')

# Download file về máy
from google.colab import files
files.download('nlp_submission.csv')

Saved submission → nlp_submission.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>